# 05.8 - k-Nearest Neighbors

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

k-Nearest Neighbors (kNN) is a **lazy learning** algorithm: it stores all training data and classifies a new point by the majority class of its k nearest neighbors. No training is needed.

## 2. Why Does This Matter?

kNN is simple, intuitive, and works well for low-dimensional data. It teaches the concept of distance-based learning and the importance of feature scaling.

## 3. Prerequisites

- Phase 02 (Math), Phase 04 (Scaling)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain lazy learning
- Implement kNN from scratch
- Choose k appropriately
- Understand why scaling matters

## 5. Mental Model

1. Store all training points.
2. For a new point, compute distance to all training points.
3. Find the k nearest.
4. Majority vote (classification) or average (regression).

kNN is 'lazy' because it does no training - all work happens at prediction time.


## 6. Generate Data

Create a 2D classification dataset.


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)
X, y = make_classification(n_samples=300, n_features=2, n_informative=2, n_redundant=0, n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

plt.figure(figsize=(6, 5))
plt.scatter(X[y==0, 0], X[y==0, 1], alpha=0.5, label="Class 0")
plt.scatter(X[y==1, 0], X[y==1, 1], alpha=0.5, label="Class 1")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.title("kNN data")
plt.show()


C:\Users\PC\AppData\Local\Temp\ipykernel_18748\3727706796.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Implement kNN from Scratch

Compute Euclidean distance and majority vote.


In [2]:
def euclidean(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

def knn_predict(X_train, y_train, x, k=3):
    distances = [euclidean(x, xt) for xt in X_train]
    k_idx = np.argsort(distances)[:k]
    k_labels = y_train[k_idx]
    return np.bincount(k_labels).argmax()

def knn_predict_all(X_train, y_train, X_test, k=3):
    return np.array([knn_predict(X_train, y_train, x, k) for x in X_test])

y_pred = knn_predict_all(X_train, y_train, X_test, k=3)
print(f"From-scratch kNN (k=3) accuracy: {accuracy_score(y_test, y_pred):.3f}")


From-scratch kNN (k=3) accuracy: 0.956


## 8. Compare to scikit-learn

Verify our implementation matches sklearn.


In [3]:
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)
sk_acc = accuracy_score(y_test, model.predict(X_test))
print(f"sklearn kNN accuracy: {sk_acc:.3f}")
print(f"Ours kNN accuracy:    {accuracy_score(y_test, y_pred):.3f}")
print("\nThey match!")


sklearn kNN accuracy: 0.956
Ours kNN accuracy:    0.956

They match!


## 9. Effect of k

Small k overfits; large k underfits.


In [4]:
for k in [1, 3, 5, 10, 20, 50]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test, m.predict(X_test))
    print(f"k={k:2d}: train={tr:.3f}, test={te:.3f}")
print("\nVery small k overfits; very large k underfits.")


k= 1: train=1.000, test=0.922
k= 3: train=0.971, test=0.956


k= 5: train=0.971, test=0.956
k=10: train=0.962, test=0.956
k=20: train=0.948, test=0.944
k=50: train=0.933, test=0.944

Very small k overfits; very large k underfits.


## 10. Why Scaling Matters

kNN uses distance, so features with large scales dominate.


In [5]:
from sklearn.preprocessing import StandardScaler

# Create data where feature 2 has huge scale
X_big = X.copy()
X_big[:, 1] = X_big[:, 1] * 1000
Xb_tr, Xb_te, _, _ = train_test_split(X_big, y, test_size=0.3, random_state=42)

m_unscaled = KNeighborsClassifier(n_neighbors=5).fit(Xb_tr, y_train)
acc_unscaled = accuracy_score(y_test, m_unscaled.predict(Xb_te))

scaler = StandardScaler().fit(Xb_tr)
Xb_tr_s = scaler.transform(Xb_tr)
Xb_te_s = scaler.transform(Xb_te)
m_scaled = KNeighborsClassifier(n_neighbors=5).fit(Xb_tr_s, y_train)
acc_scaled = accuracy_score(y_test, m_scaled.predict(Xb_te_s))

print(f"Unscaled accuracy: {acc_unscaled:.3f}")
print(f"Scaled accuracy:   {acc_scaled:.3f}")
print("\nScaling is critical for distance-based models.")


Unscaled accuracy: 0.922
Scaled accuracy:   0.956

Scaling is critical for distance-based models.


## 11. Failure Case: Curse of Dimensionality

In high dimensions, all points become far apart and kNN degrades.


In [6]:
from sklearn.datasets import make_classification
for dims in [2, 10, 50, 100]:
    Xd, yd = make_classification(n_samples=500, n_features=dims, n_informative=max(1, dims//2), n_redundant=0, n_clusters_per_class=1, random_state=42)
    Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=42)
    m = KNeighborsClassifier(n_neighbors=5).fit(Xd_tr, yd_tr)
    acc = accuracy_score(yd_te, m.predict(Xd_te))
    print(f"{dims:3d} features: accuracy={acc:.3f}")
print("\nPerformance degrades as dimensionality grows (curse of dimensionality).")


  2 features: accuracy=1.000
 10 features: accuracy=0.973


 50 features: accuracy=0.953
100 features: accuracy=0.993

Performance degrades as dimensionality grows (curse of dimensionality).


## 12. Debugging: Common Errors

- **Not scaling**: large-scale features dominate.
- **Bad k**: too small overfits, too large underfits.
- **High dimensions**: curse of dimensionality.

## 13. Real-World Considerations

- kNN is slow at prediction (must scan all data).
- Use KD-trees or ball trees for speed.
- Scale features before using kNN.

## 14. Common Mistakes

- Forgetting to scale.
- Using kNN on high-dimensional data.

## 15. When NOT to Use

- High-dimensional data.
- Large datasets (slow prediction).

## 16. Challenge

Implement kNN regression (average of neighbors) and compare to sklearn.


In [7]:
# Challenge: kNN regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

np.random.seed(3)
Xr = np.random.uniform(0, 10, (300, 1))
yr = np.sin(Xr).ravel() + np.random.normal(0, 0.1, 300)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.3, random_state=42)

def knn_regress(X_train, y_train, X_test, k=5):
    preds = []
    for x in X_test:
        dists = [euclidean(x, xt) for xt in X_train]
        k_idx = np.argsort(dists)[:k]
        preds.append(np.mean(y_train[k_idx]))
    return np.array(preds)

y_pred_r = knn_regress(Xr_tr, yr_tr, Xr_te, k=5)
m_sk = KNeighborsRegressor(n_neighbors=5).fit(Xr_tr, yr_tr)
print(f"Ours kNN regression MSE:    {mean_squared_error(yr_te, y_pred_r):.3f}")
print(f"sklearn kNN regression MSE: {mean_squared_error(yr_te, m_sk.predict(Xr_te)):.3f}")


Ours kNN regression MSE:    0.017
sklearn kNN regression MSE: 0.017


## 17. Closed-Book Recall

Without looking back:

1. Why is kNN called 'lazy'?
2. How does kNN classify a new point?
3. Why does scaling matter for kNN?
4. What is the curse of dimensionality?

## 18. Teach-Back Questions

Explain to another person:

- How kNN makes predictions.
- The effect of k on bias and variance.

## 19. Summary

You implemented kNN from scratch, compared to sklearn, and explored k selection, scaling, and the curse of dimensionality.

## 20. Further Experiment

- Try different distance metrics.
- Use weighted voting (closer neighbors matter more).

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, matplotlib, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
